# Train ELI5 Surrogates

Trains TF-IDF + Logistic Regression surrogate models for each preset LLM in the toolkit and writes pickled bundles to `medical_llm_toolkit/eli5_surrogates/`. The Streamlit app loads these bundles so ELI5 explanations are instant for the canonical preset models.

For each `(model, task_type)` we train three surrogates that share one TF-IDF vectorizer:

- **mimic** &mdash; predicts what the LLM outputs (the actual model explanation).
- **gold** &mdash; predicts the true answer (dataset-level diagnostic; LLM-independent).
- **error** &mdash; predicts whether the LLM was wrong (failure-mode diagnostic).

## How to use

1. Run cells **Setup**, **Config**, **Load datasets**, **Helpers** in order.
2. Run any subset of the four model cells. Each one loads its own LLM, queries it on the bundled dataset, trains all three surrogates, saves the bundle, then frees GPU memory.
3. MedGemma requires a Hugging Face access token (set `HF_TOKEN` in your environment first).

Each `(model, task)` bundle is a few MB. Expected runtime per model with `N_SAMPLES=1000`: roughly 5&ndash;15 minutes on a single GPU.

## Setup

In [ ]:
import sys, os, gc, pickle, re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Make the toolkit importable whether you ran `pip install -e .` or not.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from medical_llm_toolkit.wrapper import MedicalLLMWrapper

## Config

In [ ]:
SEED = 42
N_SAMPLES = 1000          # auto-capped to dataset size (YN has ~602 rows)
NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF = 0.95
TEST_SIZE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = REPO_ROOT / "data"
OUT_DIR = REPO_ROOT / "medical_llm_toolkit" / "eli5_surrogates"
OUT_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN")  # required for gated models (e.g. MedGemma)

print(f"device: {DEVICE}")
print(f"writing surrogates to: {OUT_DIR}")
print(f"HF token set: {bool(HF_TOKEN)}")

## Load datasets

In [ ]:
mcq_df = pd.read_parquet(DATA_DIR / "mcq_df.parquet")
yn_df  = pd.read_parquet(DATA_DIR / "yn_df.parquet")
print(f"MCQ: {len(mcq_df):>5d} rows  |  YN: {len(yn_df):>5d} rows")
print(f"MCQ columns: {mcq_df.columns.tolist()}")

## Helpers

In [ ]:
# --- Prompt rendering and answer parsing ---
def extract_abcd_from_options(options_text):
    if options_text is None:
        return None
    s = str(options_text)
    matches = re.findall(r"(?is)\b([ABCD])\s*[\.\)]\s*(.*?)(?=\n\s*[ABCD]\s*[\.\)]|\Z)", s)
    if not matches or len(matches) < 4:
        return None
    d = {k.upper(): v.strip() for k, v in matches}
    return d if all(k in d for k in "ABCD") else None

def render_mcq_prompt(question, options_text):
    opts = extract_abcd_from_options(options_text)
    if not opts:
        return (
            "You are a careful medical question-answering assistant.\n"
            "Choose the single best option.\n\n"
            f"Question: {question}\n{options_text}\nAnswer: "
        )
    return (
        "You are a careful medical question-answering assistant.\n"
        "Choose the single best option.\n\n"
        f"Question: {question}\nAnswer Choices:\n"
        f"A. {opts['A']}\nB. {opts['B']}\nC. {opts['C']}\nD. {opts['D']}\nAnswer: "
    )

def render_yn_prompt(question):
    return (
        "You are a careful medical question-answering assistant.\n"
        "Answer Yes or No.\nUse A for Yes and B for No.\n\n"
        f"Question: {question}\nAnswer: "
    )

def normalize_gold_yn(val):
    g = str(val).strip().lower()
    if g in {"yes", "y", "true", "t", "1", "a"}: return "A"
    if g in {"no", "n", "false", "f", "0", "b"}: return "B"
    return None

# --- Surrogate input text (no prompt boilerplate) ---
def mcq_surrogate_text(question, options_text):
    opts = extract_abcd_from_options(options_text)
    if opts:
        base = f"{question} A {opts['A']} B {opts['B']} C {opts['C']} D {opts['D']}"
    else:
        base = f"{question} {options_text}"
    return re.sub(r"\s+", " ", str(base)).strip()

def yn_surrogate_text(question):
    return re.sub(r"\s+", " ", f"{question} (A=Yes, B=No)").strip()

# --- Data collection: query LLM in answer_only mode for each row ---
def collect_predictions(wrapper, df_subset, task_type, n_samples):
    valid = ["A", "B", "C", "D"] if task_type == "mcq" else ["A", "B"]
    sub = df_subset.sample(n=min(n_samples, len(df_subset)), random_state=SEED).reset_index(drop=True)

    wrapper.set_task(task_type)
    wrapper.set_mode("answer_only")

    texts, golds, preds, wrongs = [], [], [], []
    for r in tqdm(sub.to_dict("records"), desc=f"{wrapper.model_name} [{task_type}]"):
        if task_type == "mcq":
            gold = str(r["answer_label"]).strip().upper()
            if gold not in valid:
                continue
            prompt = render_mcq_prompt(r["question"], r["options"])
            text = mcq_surrogate_text(r["question"], r["options"])
        else:
            gold = normalize_gold_yn(r["answer_label"])
            if gold not in valid:
                continue
            prompt = render_yn_prompt(r["question"])
            text = yn_surrogate_text(r["question"])

        try:
            wrapper.generate(prompt)
            pred = (wrapper.last_answer or "").strip().upper()
        except Exception as e:
            print(f"  [skip] generation error: {e}")
            continue
        if pred not in valid:
            continue

        texts.append(text)
        golds.append(gold)
        preds.append(pred)
        wrongs.append(int(pred != gold))

    return texts, golds, preds, wrongs

# --- Surrogate training: shared TF-IDF, three classifiers ---
def train_surrogate_bundle(texts, golds, preds, wrongs):
    n = len(texts)
    if n < 30:
        raise RuntimeError(f"Not enough samples to train (got {n}, need >=30)")

    idx = np.arange(n)
    strat = preds if len(set(preds)) > 1 else None
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE, random_state=SEED, stratify=strat)
    Xtr_text = [texts[i] for i in idx_tr]
    Xte_text = [texts[i] for i in idx_te]

    vec = TfidfVectorizer(ngram_range=NGRAM_RANGE, min_df=MIN_DF, max_df=MAX_DF)
    Xtr = vec.fit_transform(Xtr_text)
    Xte = vec.transform(Xte_text)
    print(f"  vocab size: {len(vec.get_feature_names_out())}")

    surrogates = {}
    for kind, y_all in (("mimic", preds), ("gold", golds), ("error", wrongs)):
        ytr = [y_all[i] for i in idx_tr]
        yte = [y_all[i] for i in idx_te]
        if len(set(ytr)) < 2:
            print(f"  [warn] skipping '{kind}': only one class in training labels")
            continue
        clf = LogisticRegression(max_iter=4000)
        clf.fit(Xtr, ytr)
        score = float(clf.score(Xte, yte))
        surrogates[kind] = {
            "clf": clf,
            "labels": [str(c) for c in clf.classes_],
            "heldout_score": score,
        }
        print(f"  {kind:5s} heldout score: {score:.3f}")

    return {
        "vectorizer": vec,
        "surrogates": surrogates,
        "vocab_size": int(len(vec.get_feature_names_out())),
        "n_train": int(len(idx_tr)),
        "n_test": int(len(idx_te)),
    }

# --- File naming and persistence ---
def model_id_to_filename(model_id, task_type):
    safe = model_id.replace("/", "__")
    return f"{safe}__{task_type}.pkl"

def save_bundle(bundle, model_id, task_type, n_samples):
    out_path = OUT_DIR / model_id_to_filename(model_id, task_type)
    payload = {
        "schema_version": 1,
        "model_id": model_id,
        "task_type": task_type,
        "n_samples_requested": int(n_samples),
        "trained_at": datetime.utcnow().isoformat() + "Z",
        **bundle,
    }
    with open(out_path, "wb") as f:
        pickle.dump(payload, f)
    print(f"  saved {out_path.name} ({out_path.stat().st_size / 1024:.1f} KB)")
    return out_path

# --- GPU cleanup between models ---
def free_gpu(wrapper):
    if wrapper is not None:
        try: wrapper.model.cpu()
        except Exception: pass
        try: del wrapper.model
        except Exception: pass
        try: del wrapper.tokenizer
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Top-level driver: train all configured tasks for one model ---
def train_for_model(model_id, hf_token=None, torch_dtype=None, tasks=("mcq", "yn")):
    print(f"\n{'='*64}\n  {model_id}\n{'='*64}")
    wrapper = MedicalLLMWrapper(model_id, device=DEVICE, token=hf_token, torch_dtype=torch_dtype)
    try:
        for task_type in tasks:
            df_sub = mcq_df if task_type == "mcq" else yn_df
            if len(df_sub) == 0:
                print(f"  [skip] no rows for task '{task_type}'")
                continue
            print(f"\n--- {task_type.upper()} ---")
            texts, golds, preds, wrongs = collect_predictions(wrapper, df_sub, task_type, N_SAMPLES)
            print(f"  collected {len(texts)} valid (text, gold, pred) rows")
            try:
                bundle = train_surrogate_bundle(texts, golds, preds, wrongs)
                save_bundle(bundle, model_id, task_type, N_SAMPLES)
            except Exception as e:
                print(f"  [error] training/saving failed: {e}")
    finally:
        free_gpu(wrapper)
    print(f"\n  done with {model_id}")

## Apollo-2B

Small open model, fast to run. No HF token required.

In [ ]:
train_for_model("FreedomIntelligence/Apollo-2B")

## MedGemma-4B

Gated model &mdash; you must accept the license on the Hugging Face model page and set `HF_TOKEN` in your environment **before launching this notebook**. The wrapper auto-forces `float32` for MedGemma.

In [ ]:
train_for_model("google/medgemma-4b-it", hf_token=HF_TOKEN, torch_dtype=torch.float32)

## BioMistral-7B

Larger model &mdash; expect a longer wall-clock time per forward pass. No HF token required.

In [ ]:
train_for_model("BioMistral/BioMistral-7B")

## BioMedLM

Stanford CRFM domain-specific LM. No HF token required.

In [ ]:
train_for_model("stanford-crfm/BioMedLM")

## Verify saved bundles

In [ ]:
for p in sorted(OUT_DIR.glob("*.pkl")):
    with open(p, "rb") as f:
        b = pickle.load(f)
    kinds = list(b.get("surrogates", {}).keys())
    scores = {k: round(b["surrogates"][k]["heldout_score"], 3) for k in kinds}
    print(f"{p.name:60s}  vocab={b.get('vocab_size','?'):>5}  n_train={b.get('n_train','?')}  scores={scores}")